In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_6.py ──
"""
Shared infrastructure for MLFP03 Exercise 6 — Interpretability and Fairness.

Contains: Singapore credit scoring data load, LightGBM model training,
TreeSHAP explainer setup, output directory, and common helper utilities.

Technique-specific code (permutation importance loops, LIME wrappers,
fairness audit reports) lives in the per-technique files under
`modules/mlfp03/solutions/ex_6/`.

Import pattern (solutions and local both):

"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl

import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input



# ════════════════════════════════════════════════════════════════════════
# PATHS / CONSTANTS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp03_ex6_interpretability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Singapore credit scoring: Monetary Authority of Singapore (MAS) requires
# explainability for credit decisions under the Model Risk Management
# guideline. This dataset simulates a retail-bank default prediction task
# used throughout MLFP02/MLFP03.
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"
RANDOM_SEED = 42

# Protected attribute candidates we audit for disparate impact.
PROTECTED_CANDIDATES: list[str] = ["age", "gender", "ethnicity", "marital_status"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOAD + MODEL TRAIN
# ════════════════════════════════════════════════════════════════════════

# Populated on first call so every technique file sees the same split.
_CACHE: dict[str, Any] = {}


def load_credit_scoring() -> dict[str, Any]:
    """Load the Singapore credit scoring dataset and run the M3 preprocessing
    pipeline. Returns a dict with X_train, y_train, X_test, y_test, feature_names.

    The return value is cached so repeated calls from different technique
    files re-use the same split (essential for interpretability comparisons).
    """
    if _CACHE:
        return _CACHE

    loader = MLFPDataLoader()
    credit: pl.DataFrame = loader.load(DATASET_MODULE, DATASET_FILE)

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target=TARGET_COLUMN,
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_columns = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    feature_names: list[str] = col_info["feature_columns"]

    _CACHE.update(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        feature_names=feature_names,
    )
    return _CACHE


def train_credit_model() -> dict[str, Any]:
    """Train the LightGBM credit default model. Cached per-process.

    Returns a dict with model, y_proba, y_pred, auc, and all data from
    `load_credit_scoring()`.
    """
    if "model" in _CACHE:
        return _CACHE

    data = load_credit_scoring()
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]

    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
        random_state=RANDOM_SEED,
        verbose=-1,
    )
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_proba)

    _CACHE.update(model=model, y_proba=y_proba, y_pred=y_pred, auc=auc)
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# SHAP EXPLAINER
# ════════════════════════════════════════════════════════════════════════


def build_shap_explainer() -> dict[str, Any]:
    """Construct the TreeSHAP explainer and compute SHAP values for X_test.

    Returns the full bundle: model, data, explainer, shap_vals, expected_value.
    """
    if "shap_vals" in _CACHE:
        return _CACHE

    bundle = train_credit_model()
    explainer = shap.TreeExplainer(bundle["model"])
    shap_values = explainer.shap_values(bundle["X_test"])

    # TreeSHAP for binary classifiers may return [class_0, class_1]
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    expected_value = (
        explainer.expected_value[1]
        if isinstance(explainer.expected_value, list)
        else explainer.expected_value
    )

    _CACHE.update(
        explainer=explainer,
        shap_vals=shap_vals,
        expected_value=expected_value,
    )
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# REUSABLE UTILITIES
# ════════════════════════════════════════════════════════════════════════


def rank_features_by_mean_abs_shap(
    shap_vals: np.ndarray, feature_names: list[str]
) -> list[tuple[str, float]]:
    """Return [(feature, mean_abs_shap), ...] sorted descending."""
    mean_abs = np.abs(shap_vals).mean(axis=0)
    return sorted(zip(feature_names, mean_abs), key=lambda x: x[1], reverse=True)


def feature_index(feature_names: list[str], name: str) -> int:
    """Lookup a feature column index by name, raising a clear error."""
    if name not in feature_names:
        raise KeyError(
            f"Feature '{name}' not found. Available: {feature_names[:10]}..."
        )
    return feature_names.index(name)


def synthetic_group_split(
    X: np.ndarray, feature_idx: int = 0
) -> tuple[np.ndarray, np.ndarray, float]:
    """Split X into two groups on a median cut of `feature_idx`.

    Returns (group_a_mask, group_b_mask, median_value).
    Used as a fallback when no protected attribute is present in features.
    """
    vals = X[:, feature_idx]
    median_val = float(np.median(vals))
    group_a = vals <= median_val
    group_b = ~group_a
    return group_a, group_b, median_val


def print_section(title: str, char: str = "=") -> None:
    """Print a standardised section banner."""
    line = char * 70
    print(f"\n{line}")
    print(f"  {title}")
    print(line)




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 6.2: From-Scratch Permutation Importance
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement permutation importance from scratch (no sklearn.inspection)
#   - Measure feature importance MODEL-AGNOSTICALLY
#   - Quantify estimator variance via repeated shuffles (mean +/- std)
#   - Compare the permutation ranking against SHAP for the same model
#   - Understand why correlated features trip up permutation but not SHAP
#   - Apply: OCBC retail-branch KPI audit on a non-tree challenger model
#
# PREREQUISITES: 01_shap_global.py
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — why permutation is model-agnostic
#   2. Build — permutation_importance_manual() from scratch
#   3. Train — MEASURE the trained model
#   4. Visualise — ranking + SHAP vs permutation top-10 overlap
#   5. Apply — OCBC champion/challenger monitoring
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from dotenv import load_dotenv
from sklearn.metrics import f1_score, roc_auc_score


load_dotenv()



## THEORY — Permutation Importance as a Model-Agnostic Probe


In [ ]:
# Algorithm:
#   1. Score the model on the test set (baseline)
#   2. For each feature j: shuffle column j, rescore, compute drop
#   3. Repeat K times, average for stability
#
# SHAP handles correlated features correctly via the Shapley axioms.
# Permutation under-reports correlated-feature importance.



## TASK 2 — BUILD the from-scratch permutation importance function


Compute permutation importance from scratch.


In [ ]:
def permutation_importance_manual(
    model,
    X: np.ndarray,
    y: np.ndarray,
    feature_names: list[str],
    n_repeats: int = 5,
    scoring: str = "roc_auc",
    seed: int = 42,
) -> tuple[float, dict[str, dict[str, float]]]:
    rng = np.random.default_rng(seed)

    # TODO: compute baseline score (predict_proba -> roc_auc_score)
    y_proba = ____
    if scoring == "roc_auc":
        baseline = ____
    else:
        baseline = f1_score(y, model.predict(X))

    importances: dict[str, dict[str, float]] = {}
    for feat_idx, feat_name in enumerate(feature_names):
        drops: list[float] = []
        for _ in range(n_repeats):
            X_shuffled = X.copy()
            # TODO: shuffle column feat_idx in place using rng.permutation(...)
            X_shuffled[:, feat_idx] = ____
            y_p_shuffled = model.predict_proba(X_shuffled)[:, 1]
            if scoring == "roc_auc":
                shuffled_score = roc_auc_score(y, y_p_shuffled)
            else:
                shuffled_score = f1_score(y, (y_p_shuffled >= 0.5).astype(int))
            # TODO: append (baseline - shuffled_score) to drops
            ____
        importances[feat_name] = {
            "mean": float(np.mean(drops)),
            "std": float(np.std(drops)),
        }

    return float(baseline), importances



## TASK 3 — measure the already-trained model


In [ ]:
bundle = build_shap_explainer()
model = bundle["model"]
X_test = bundle["X_test"]
y_test = bundle["y_test"]
feature_names: list[str] = bundle["feature_names"]
shap_vals = bundle["shap_vals"]

print_section("From-Scratch Permutation Importance on Credit Model")
# TODO: call permutation_importance_manual with n_repeats=5, scoring="roc_auc"
baseline_score, perm_imp = ____
print(f"Baseline AUC-ROC: {baseline_score:.4f}")



## TASK 4 — VISUALISE ranking + compare with SHAP


In [ ]:
# TODO: sort perm_imp dict by the "mean" key descending -> list of (name, vals) tuples
perm_ranking = ____

print(f"\n{'Rank':>4} {'Feature':<30} {'Imp (mean)':>12} {'+/- std':>10}")
print("─" * 58)
for rank, (name, vals) in enumerate(perm_ranking[:15], 1):
    print(f"{rank:>4} {name:<30} {vals['mean']:>12.4f} {vals['std']:>10.4f}")

shap_ranking = rank_features_by_mean_abs_shap(shap_vals, feature_names)

print_section("SHAP vs Permutation — Top 10 Overlap", char="─")
shap_top10 = {n for n, _ in shap_ranking[:10]}
perm_top10 = {n for n, _ in perm_ranking[:10]}
# TODO: compute set intersection of shap_top10 and perm_top10
overlap = ____
print(f"Overlap size: {len(overlap)} / 10")
print(f"Shared features: {sorted(overlap)}")



### Checkpoint


In [ ]:
assert len(perm_imp) == len(feature_names), "Task 4: all features must be permuted"
assert (
    perm_ranking[0][1]["mean"] > 0
), "Task 4: top feature must have positive importance"
print("\n[ok] Checkpoint — permutation ranking matches SHAP top-10 structure\n")



## TASK 5 — APPLY: OCBC Challenger-Model Monitoring


In [ ]:
# SCENARIO: OCBC Bank (Singapore) runs a champion/challenger architecture:
# production tree + quarterly DNN challenger on the same features. They
# need feature-importance monitoring that produces COMPARABLE numbers
# across both model types.
#
# BUSINESS IMPACT: A unified permutation dashboard would have caught the
# 2024 income_verification drift 6 weeks earlier — S$2.1M in avoided
# write-offs against a S$16,000 implementation cost. ~130x ROI.



## REFLECTION


[x] Implemented permutation importance from scratch
  [x] Estimated per-feature importance with mean +/- std
  [x] Compared permutation and SHAP rankings on the same credit model
  [x] Explained why correlated features trip up permutation but not SHAP
  [x] Mapped the pipeline to OCBC champion/challenger monitoring

  KEY INSIGHT: Permutation is the lingua franca of cross-architecture
  feature monitoring. Use SHAP for exact attribution on trees,
  permutation for cross-model comparability.

  Next: 03_lime_local.py


In [ ]:
print_section("WHAT YOU'VE MASTERED")
print(
)

